# ILT: Exercise 1 — Training and Evaluating a Simple TnT-like POS Tagger

In the lecture we discussed criteria for a good tagger, including adaptability, efficiency, and reusability.
What does that mean in practice? A statistical tagger such as the TnT tagger can be trained for any language and tagset given a corpus annotated with POS tags. Here we demonstrate two implementations of the TnT approach:
 - The [NLTK TnT reimplementation](https://www.nltk.org/_modules/nltk/tag/tnt.html): training and tagging is wrapped with a [small wrapper script](ex03/tagger.py). This tagger is relatively slow during tagging, but allows inspecting the Python code.

- The [hunpos tagger](https://github.com/mivoq/hunpos): builds a binary model file during training with a separate training program. This model file is then applied efficiently by the tagger runtime. For [our UD pipeline web demo](https://pub.cl.uzh.ch/users/siclemat/lehre/ecl1/ud-de-hunpos-maltparser/html/), we use hunpos because it is fast and supports unknown-word guessing via suffix models.

Explanation: Lines starting with `!` are Unix shell commands executed from this notebook. You can also execute them directly in a terminal.


## (0) Working directory and files
Be sure that your working directory contains the corpus (`de_gsd-ud-2.17.tts`), the Python program `tagger.py`, as well as the evaluation script `tag-eval.perl`.
When opening this notebook, the working directory is already `ilt-notebooks/ex01`, where all required files are located.
To navigate or inspect your files, you can use `%pwd` and `!ls -l`.

In [ ]:
%pwd

In [ ]:
! ls -l

## (1) 9/10 of the corpus will be our training data
Note: While official train/dev/test splits exist for this treebank, in this exercise we demonstrate how to create a 90%/10% split directly from the unified corpus.

In [ ]:
!head -n 277511 de_gsd-ud-2.17.tts > training.tts

## (1a) Look at training data
Which tokens with the string "bestimmt" do have which UPOS tags?

In [ ]:
!grep -C 3 bestimmt training.tts

## (2) 1/10 of our corpus will be our test data
Copy the last 30834 lines into `test.tts`

In [ ]:
! tail -n 30834 de_gsd-ud-2.17.tts > test.tts

## (3) create an evaluation corpus by removing the tags from our test corpus


In [ ]:
! cut -f 1 test.tts > test.txt

## (4) Train a model with hunpos training tool

In [ ]:
! hunpos-train training.mod < training.tts

## (5) Tag the test set with the hunpos tagger and the trained model

In [ ]:
! hunpos-tag training.mod < test.txt > result.tts

## (4b/5b)  Training and tagging with NLTK tnt tagger:
Train the tagger with training.tts. Tag the evaluation file and write the results in result.tts (this can take a few minutes ... be patient). Some status information will appear on standard error.


In [ ]:
! python3 tagger.py training.tts < test.txt > result-nltk.tts

## (6) Evaluation of hunpos tagger output
Evaluate the *hunpos tagger result* with a nice confusion matrix by comparing our tagger output `result.tts` with the test corpus `test.tts`

In [ ]:
! perl tag-eval.perl -k test.tts -e result.tts 

## (6b) Evaluation of NLTK tagger output
Evaluate the NLTK tnt tagger:

In [ ]:
! tag-eval.perl -e result-nltk.tts -k test.tts

Our simple NLTK-based POS Tagger does not guess unknown words... See yourself...

In [ ]:
!grep -C 3 Unk result-nltk.tts

## (7) Testing with your own tokenized input

In [ ]:
! echo "Dies ist ein Satz !" | tr " " "\n" | hunpos-tag training.mod 

## Control question

What happens if you train over the whole corpus and evaluate afterwards? (Just substitute `training.tts` in the training and tagging command with the whole corpus `de_gsd-ud-2.17.tts`)

Supplement: How to compute the length of your training and test corpus automatically according to your split proportions? Note: This simple method can cut in the middle of a sentence...


In [ ]:
%%bash
export corpuslen=$(wc -l <de_gsd-ud-2.17.tts )
export testlen=$(($corpuslen / 10 ))
export trainlen=$(($corpuslen - $testlen))
echo Corpus lines $corpuslen test lines $testlen training lines $trainlen
